# Datathon Task 1 V4 — Purged Day OOF + Phase/Clause Diffusion

V4 memperbaiki akar gap local 26.22 vs public 38.639. Dua hari tail validation V3 ternyata jauh lebih mudah daripada keseluruhan periode: mean-last-3 MSE 35.63 di tail versus 49.19 pada seluruh train. Karena itu V4 tidak lagi mengoptimalkan tail score.

Alur notebook: audit validation bias → purged day-fold split → event clause parsing → spatial graph → tiga neural ablation → natural OOF evaluation → cross-fitted ensemble → full refit → submission.

**Runtime:** Kaggle GPU T4, internet off. Estimasi full mode 25–60 menit tergantung kandidat yang mendapat bobot ensemble. Semua loop panjang memakai `tqdm`.


## 1. Hipotesis dan guardrail

Tiga ablation menguji penambahan signal secara bertahap:

1. `global_tcn_diffusion`: temporal gated convolution + directed physical graph + global FiLM.
2. `phase_tcn_diffusion`: model pertama + phase waktu harian yang diinferensikan dari history; timestamp test tidak digunakan.
3. `phase_clause_geo_diffusion`: model kedua + clause-level event-to-road attention + geometry-neighbor diffusion.

Semua score utama berasal dari tiga day-block folds yang mencakup seluruh periode secara natural. Window tidak boleh melintasi batas hari, sehingga train dan validation fold tidak berbagi observation. Epoch count fixed agar validation fold tidak sekaligus menjadi early-stopping selector. Ensemble dievaluasi cross-fit: bobot untuk satu fold dipelajari dari fold lain.


In [ ]:
from __future__ import annotations
import gc, hashlib, importlib, json, math, os, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm, trange
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

BASE_SEED=42; HORIZONS=(5,10,15); HISTORY=15; N_ROADS=1260; STEPS_PER_DAY=360
ANCHOR_IDX=np.arange(0,N_ROADS,10,dtype=np.int64); GLOBAL_DIM=HISTORY*3+len(ANCHOR_IDX)
PUBLIC_V3=38.639; V3_LOCAL=26.21852
KAGGLE_INPUT=Path('/kaggle/input/competitions/datathon-task-1/dataset-task1')
LOCAL_CANDIDATES=[Path.cwd()/'dataset-task1']+[p/'dataset-task1' for p in Path.cwd().parents]
LOCAL_INPUT=next((p for p in LOCAL_CANDIDATES if p.exists()),LOCAL_CANDIDATES[0])
DATA_DIR=KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT
IS_KAGGLE=KAGGLE_INPUT.exists(); FAST_MODE=os.getenv('DATATON_FAST_MODE','0')=='1'
OUTPUT_DIR=Path('/kaggle/working') if IS_KAGGLE else DATA_DIR.parent/'tmp/jupyter-notebook/v4-smoke'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

N_FOLDS=3 if not FAST_MODE else 2; CV_STRIDE=2 if not FAST_MODE else 60; FULL_STRIDE=1 if not FAST_MODE else 60
CV_EPOCHS=8 if not FAST_MODE else 1; FINAL_EPOCHS=8 if not FAST_MODE else 1
FINAL_SEEDS=[42,2026,3407] if not FAST_MODE else [42]
HIDDEN=48 if not FAST_MODE else 12; CLAUSE_DIM=32 if not FAST_MODE else 8
MAX_CLAUSES=36 if not FAST_MODE else 16; LOCATION_BUCKETS=2048 if not FAST_MODE else 256
BATCH_SIZE=12 if torch.cuda.is_available() else (2 if FAST_MODE else 3); NUM_WORKERS=0
DEVICE=torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))

if not hasattr(torch,'_utils'): torch._utils=importlib.import_module('torch._utils')
probe=nn.Parameter(torch.tensor([1.])); probe_opt=torch.optim.AdamW([probe],lr=1e-3); (probe**2).sum().backward(); probe_opt.step(); del probe,probe_opt
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(BASE_SEED)
print('Optimizer preflight OK | torch',torch.__version__)
print({'data':str(DATA_DIR),'output':str(OUTPUT_DIR),'device':str(DEVICE),'fast_mode':FAST_MODE,'folds':N_FOLDS,'cv_stride':CV_STRIDE,'epochs':CV_EPOCHS,'batch':BATCH_SIZE})
assert DATA_DIR.exists()


## 2. Data load dan bukti validation bias

Grafik harian di bawah adalah alasan validation protocol diganti. Metric antarhari sangat nonstationary; satu tail period tidak cukup untuk ranking model.


In [ ]:
def load_json_values(p):
    with Path(p).open() as f:return list(json.load(f).values())
speed_paths=sorted((DATA_DIR/'train').glob('train_speed_*.npy')); text_paths=sorted((DATA_DIR/'train').glob('train_text_*.json'))
train_speeds=[np.load(p,mmap_mode='r') for p in speed_paths]; train_texts=[load_json_values(p) for p in text_paths]
test_hist=np.load(DATA_DIR/'test/test_X_hist.npy'); test_texts=load_json_values(DATA_DIR/'test/test_texts.json')
adjacency=np.load(DATA_DIR/'static/matrix.npy'); roads_meta=json.load(open(DATA_DIR/'static/Roads1260.json')); sample_submission=pd.read_csv(DATA_DIR/'sample_submission.csv')
assert test_hist.shape==(540,HISTORY,N_ROADS) and adjacency.shape==(N_ROADS,N_ROADS)

def mean3_score(z,times):
    if not len(times): return np.repeat(np.nan,3)
    sse=np.zeros(3,np.float64); count=0
    for start in range(0,len(times),256):
        ts=times[start:start+256]; base=np.stack([z[t-2:t+1].mean(0) for t in ts]).astype(np.float32)
        for i,h in enumerate(HORIZONS): sse[i]+=np.sum((np.asarray(z[ts+h],np.float32)-base)**2,dtype=np.float64)
        count+=len(ts)*N_ROADS
    return sse/count

daily=[]
for b,z in enumerate(train_speeds):
    for day in range(int(np.ceil(len(z)/STEPS_PER_DAY))):
        lo=max(HISTORY-1,day*STEPS_PER_DAY); hi=min((day+1)*STEPS_PER_DAY,len(z)-max(HORIZONS))
        if hi<=lo: continue
        s=mean3_score(z,np.arange(lo,hi)); daily.append({'block':f'm{b+1}','day':day,'h5':s[0],'h10':s[1],'h15':s[2],'mean':s.mean()})
daily_df=pd.DataFrame(daily)
display(daily_df.groupby('block').agg(days=('day','count'),mean=('mean','mean'),std=('mean','std'),minimum=('mean','min'),maximum=('mean','max')).round(3))
fig,ax=plt.subplots(1,2,figsize=(14,4))
for name,g in daily_df.groupby('block'): ax[0].plot(g.day,g['mean'],marker='o',ms=3,label=name)
ax[0].set(title='Mean-last-3 MSE changes sharply by day',xlabel='day',ylabel='MSE'); ax[0].legend()
ax[1].hist(daily_df.query("block=='m1'")['mean'],alpha=.6,bins=12,label='m1'); ax[1].hist(daily_df.query("block=='m2'")['mean'],alpha=.6,bins=12,label='m2'); ax[1].set(title='Daily difficulty distribution',xlabel='MSE'); ax[1].legend(); plt.tight_layout(); plt.show()
print('V3 public/local:',PUBLIC_V3,V3_LOCAL,'| conclusion: old tail validation is not a trustworthy absolute benchmark.')


## 3. Purged day-block folds

Setiap origin hanya dipakai jika seluruh 15-step history dan target h15 berada dalam hari yang sama. Fold assignment `day % K` menyebarkan validation ke awal, tengah, dan akhir periode. Natural OOF aggregation otomatis memberi m1/m2 proporsi yang hampir sama dengan test.


In [ ]:
def day_safe_times(z,stride=1):
    parts=[]
    for day in range(int(np.ceil(len(z)/STEPS_PER_DAY))):
        start=day*STEPS_PER_DAY; end=min((day+1)*STEPS_PER_DAY,len(z))
        lo=start+HISTORY-1; hi=end-max(HORIZONS)
        if hi>lo: parts.append(np.arange(lo,hi,stride,dtype=np.int32))
    return np.concatenate(parts) if parts else np.empty(0,np.int32)

all_cv_times=[day_safe_times(z,CV_STRIDE) for z in train_speeds]
fold_val_times=[]; fold_train_times=[]
for fold in range(N_FOLDS):
    va=[t[(t//STEPS_PER_DAY)%N_FOLDS==fold] for t in all_cv_times]
    tr=[t[(t//STEPS_PER_DAY)%N_FOLDS!=fold] for t in all_cv_times]
    if FAST_MODE:
        tr=[x[-min(14,len(x)):] for x in tr]; va=[x[-min(8,len(x)):] for x in va]
    fold_train_times.append(tr); fold_val_times.append(va)

fold_table=[]
for f,(tr,va) in enumerate(zip(fold_train_times,fold_val_times)):
    fold_table.append({'fold':f,'train_m1':len(tr[0]),'train_m2':len(tr[1]),'val_m1':len(va[0]),'val_m2':len(va[1]),'val_total':sum(map(len,va))})
display(pd.DataFrame(fold_table))

fig,ax=plt.subplots(figsize=(14,2.8))
for b,z in enumerate(train_speeds):
    days=np.arange(int(np.ceil(len(z)/STEPS_PER_DAY))); ax.scatter(days,np.full_like(days,b),c=days%N_FOLDS,cmap='tab10',s=65)
ax.set_yticks([0,1],['m1','m2']); ax.set(title='Day-block fold assignment',xlabel='day'); plt.show()


## 4. Clause-level event representation

V3 memadatkan seluruh dokumen menjadi satu vector. V4 memecah text berdasarkan titik, mengenali event type, dan meng-hash location phrase secara deterministik. Hashing tidak perlu fit pada validation/test dan tetap menangani lokasi baru. Road query kemudian melakukan attention ke clause relevan.


In [ ]:
EVENT_NAMES=['padding','closure','construction','accident','announcement','prohibit','closure+construction','other']
def event_type_id(s):
    s=s.lower(); closure='closure' in s; construction='construction' in s
    if closure and construction:return 6
    if 'accident' in s:return 3
    if 'announcement' in s:return 4
    if 'prohibit' in s:return 5
    if construction:return 2
    if closure:return 1
    return 7
def stable_location_id(s):
    location=s.lower().split(' on ',1)[1].strip() if ' on ' in s.lower() else s.lower().strip()
    digest=hashlib.blake2b(location.encode(),digest_size=4).digest(); return int.from_bytes(digest,'little')%LOCATION_BUCKETS+1
def encode_documents(texts,desc):
    loc=np.zeros((len(texts),MAX_CLAUSES),np.int64); typ=np.zeros_like(loc); counts=np.zeros(len(texts),np.int16)
    for i,doc in enumerate(tqdm(texts,desc=desc,leave=False)):
        clauses=[x.strip() for x in doc.split('.') if x.strip()][:MAX_CLAUSES]; counts[i]=len(clauses)
        for j,c in enumerate(clauses): loc[i,j]=stable_location_id(c); typ[i,j]=event_type_id(c)
    return loc,typ,counts

train_clause=[]
for b,texts in enumerate(train_texts): train_clause.append(encode_documents(texts,f'parse train m{b+1}'))
test_loc,test_type,test_clause_count=encode_documents(test_texts,'parse test events')
type_counts=np.bincount(np.concatenate([x[1].ravel() for x in train_clause]),minlength=len(EVENT_NAMES))[1:]
clause_report=pd.DataFrame({'event_type':EVENT_NAMES[1:],'count':type_counts}); display(clause_report)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].bar(clause_report.event_type,clause_report['count']); ax[0].tick_params(axis='x',rotation=25); ax[0].set(title='Parsed event types')
ax[1].hist(train_clause[0][2],alpha=.55,label='m1'); ax[1].hist(train_clause[1][2],alpha=.55,label='m2'); ax[1].hist(test_clause_count,alpha=.45,label='test'); ax[1].set(title='Clauses per document'); ax[1].legend(); plt.tight_layout(); plt.show()
print('Clause median m1/m2/test:',np.median(train_clause[0][2]),np.median(train_clause[1][2]),np.median(test_clause_count))


## 5. Directed physical graph dan geometry neighbors

Physical adjacency menangkap network connectivity. Geometry kNN menambah local spatial relation ketika adjacency tidak langsung menghubungkan ruas yang berdekatan. Edge gates dipelajari hanya pada edge yang tersedia, sehingga jauh lebih ringan daripada dense adaptive adjacency 1260×1260.


In [ ]:
src_np,dst_np=np.nonzero(adjacency!=0); keep=src_np!=dst_np; src_np,dst_np=src_np[keep].astype(np.int64),dst_np[keep].astype(np.int64)
centroids=[]; metadata=[]
for components in roads_meta:
    coords=[]; total_len=0.; classes=[]; forms=[]
    for seg in components:
        xy=np.asarray(seg['coordList'],dtype=np.float64).reshape(-1,2); coords.append(xy); total_len+=float(seg.get('length',0)); classes.append(float(seg.get('roadclass',0))); forms.append(float(seg.get('formway',0)))
    xy=np.concatenate(coords); centroids.append(xy.mean(0)); metadata.append([np.log1p(total_len),np.mean(classes),np.mean(forms)])
centroids=np.asarray(centroids); metadata=np.asarray(metadata,np.float32)
geo_xy=centroids.copy(); geo_xy[:,0]*=np.cos(np.deg2rad(geo_xy[:,1].mean()))
geo_nn=NearestNeighbors(n_neighbors=7).fit(geo_xy); _,neighbors=geo_nn.kneighbors(geo_xy)
geo_src_np=np.repeat(np.arange(N_ROADS),6).astype(np.int64); geo_dst_np=neighbors[:,1:].reshape(-1).astype(np.int64)

graph_report=pd.DataFrame([{'relation':'physical directed','edges':len(src_np)},{'relation':'geometry kNN','edges':len(geo_src_np)}]); display(graph_report)
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].hist(np.bincount(dst_np,minlength=N_ROADS),bins=25); ax[0].set(title='Physical in-degree')
ax[1].scatter(centroids[:,0],centroids[:,1],s=4,alpha=.5); ax[1].set(title='Road centroids',xlabel='longitude',ylabel='latitude'); plt.tight_layout(); plt.show()


## 6. Dataset, fold-local normalization, dan model family

Temporal encoder memakai gated dilated convolutions (dilation 1/2/4). Global history menghasilkan FiLM parameters dan optional soft daily phase. Diffusion blocks memisahkan incoming, outgoing, dan geometry messages dengan learnable edge gates.


In [ ]:
def global_raw_from_windows(w): return np.concatenate([w.mean(2),w.std(2),(w==0).mean(2),w[:,-1,ANCHOR_IDX]],1).astype(np.float32)
def fit_fold_context(times_by_block,batch=128):
    latest=[]; globals_=[]
    for z,times in zip(train_speeds,times_by_block):
        for start in range(0,len(times),batch):
            ts=times[start:start+batch]; w=np.stack([z[t-HISTORY+1:t+1] for t in ts]).astype(np.float32)
            latest.append(w[:,-1]); globals_.append(global_raw_from_windows(w))
    latest=np.concatenate(latest); g=np.concatenate(globals_); scaler=StandardScaler().fit(g)
    return latest.mean(0).astype(np.float32),(latest.std(0)+3.).astype(np.float32),scaler

class TrafficDataset(Dataset):
    def __init__(self,times_by_block): self.index=[(b,int(t)) for b,times in enumerate(times_by_block) for t in times]
    def __len__(self):return len(self.index)
    def __getitem__(self,i):
        b,t=self.index[i]; z=train_speeds[b]; hist=np.asarray(z[t-HISTORY+1:t+1],np.float32).T.copy(); target=np.stack([z[t+h] for h in HORIZONS],-1).astype(np.float32)
        loc,typ,_=train_clause[b]; angle=2*np.pi*(t%STEPS_PER_DAY)/STEPS_PER_DAY; phase=np.array([np.sin(angle),np.cos(angle)],np.float32)
        return torch.from_numpy(hist),torch.from_numpy(loc[t]),torch.from_numpy(typ[t]),torch.from_numpy(target),torch.from_numpy(phase),torch.tensor(b)

class GatedTemporalEncoder(nn.Module):
    def __init__(self,hidden):
        super().__init__(); self.input=nn.Conv1d(3,hidden,1); self.dilations=(1,2,4)
        self.filters=nn.ModuleList([nn.Conv1d(hidden,hidden,2,dilation=d) for d in self.dilations]); self.gates=nn.ModuleList([nn.Conv1d(hidden,hidden,2,dilation=d) for d in self.dilations]); self.residuals=nn.ModuleList([nn.Conv1d(hidden,hidden,1) for _ in self.dilations]); self.dropout=nn.Dropout(.1)
    def forward(self,x):
        x=self.input(x)
        for d,cf,cg,cr in zip(self.dilations,self.filters,self.gates,self.residuals):
            xp=F.pad(x,(d,0)); z=torch.tanh(cf(xp))*torch.sigmoid(cg(xp)); x=x+self.dropout(cr(z))
        return x[:,:,-1]

class AdaptiveDiffusionBlock(nn.Module):
    def __init__(self,hidden,use_geo):
        super().__init__(); self.use_geo=use_geo; gate_dim=8
        self.edge_src=nn.Embedding(N_ROADS,gate_dim); self.edge_dst=nn.Embedding(N_ROADS,gate_dim)
        self.self_linear=nn.Linear(hidden,hidden); self.in_linear=nn.Linear(hidden,hidden,bias=False); self.out_linear=nn.Linear(hidden,hidden,bias=False); self.geo_linear=nn.Linear(hidden,hidden,bias=False) if use_geo else None
        self.norm=nn.LayerNorm(hidden); self.dropout=nn.Dropout(.1)
    def aggregate(self,h,src,dst):
        gate=torch.sigmoid((self.edge_src(src)*self.edge_dst(dst)).sum(-1)/math.sqrt(self.edge_src.embedding_dim)); out=torch.zeros_like(h); out.index_add_(1,dst,h[:,src]*gate[None,:,None])
        denom=torch.zeros(N_ROADS,device=h.device,dtype=h.dtype); denom.index_add_(0,dst,gate.to(h.dtype)); return out/denom.clamp_min(.1)[None,:,None]
    def forward(self,h,src,dst,geo_src,geo_dst):
        incoming=self.aggregate(h,src,dst); outgoing=self.aggregate(h,dst,src); u=self.self_linear(h)+self.in_linear(incoming)+self.out_linear(outgoing)
        if self.use_geo: u=u+self.geo_linear(self.aggregate(h,geo_src,geo_dst))
        return self.norm(h+self.dropout(F.gelu(u)))

@dataclass(frozen=True)
class ModelConfig:
    name:str; use_phase:bool; use_clause:bool; use_geo:bool; phase_weight:float=.05

class PhaseClauseDiffusionNet(nn.Module):
    def __init__(self,config,road_mean,road_std,global_scaler,hidden=HIDDEN):
        super().__init__(); self.config=config
        self.register_buffer('road_mean',torch.tensor(road_mean).view(1,N_ROADS,1)); self.register_buffer('road_std',torch.tensor(road_std).view(1,N_ROADS,1)); self.register_buffer('global_mean',torch.tensor(global_scaler.mean_,dtype=torch.float32)); self.register_buffer('global_scale',torch.tensor(global_scaler.scale_,dtype=torch.float32))
        self.register_buffer('src',torch.tensor(src_np)); self.register_buffer('dst',torch.tensor(dst_np)); self.register_buffer('geo_src',torch.tensor(geo_src_np)); self.register_buffer('geo_dst',torch.tensor(geo_dst_np))
        self.temporal=GatedTemporalEncoder(hidden); self.road_embedding=nn.Embedding(N_ROADS,12); self.road_proj=nn.Linear(12,hidden)
        self.global_encoder=nn.Sequential(nn.Linear(GLOBAL_DIM,hidden),nn.GELU(),nn.Linear(hidden,hidden),nn.GELU()); self.phase_head=nn.Linear(hidden,2)
        film_in=hidden+(2 if config.use_phase else 0); self.film=nn.Sequential(nn.Linear(film_in,hidden),nn.GELU(),nn.Linear(hidden,hidden*2))
        if config.use_clause:
            self.location_embedding=nn.Embedding(LOCATION_BUCKETS+1,CLAUSE_DIM,padding_idx=0); self.type_embedding=nn.Embedding(len(EVENT_NAMES),8,padding_idx=0); self.clause_encoder=nn.Sequential(nn.Linear(CLAUSE_DIM+8,hidden),nn.GELU(),nn.Linear(hidden,hidden)); self.clause_key=nn.Linear(hidden,16); self.clause_value=nn.Linear(hidden,hidden); self.road_query=nn.Embedding(N_ROADS,16); self.clause_scale=nn.Parameter(torch.tensor(.1))
        self.graph_blocks=nn.ModuleList([AdaptiveDiffusionBlock(hidden,config.use_geo),AdaptiveDiffusionBlock(hidden,config.use_geo)])
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(hidden,hidden//2),nn.GELU(),nn.Linear(hidden//2,1)) for _ in HORIZONS])
        for head in self.heads: nn.init.zeros_(head[-1].weight); nn.init.zeros_(head[-1].bias)
    def forward(self,hist,loc,typ):
        base=hist[:,:,-3:].mean(-1,keepdim=True); normalized=(hist-self.road_mean)/self.road_std; deviations=(hist-base)/self.road_std; zero=(hist==0).to(hist.dtype)
        temporal_in=torch.stack([normalized,deviations,zero],2).reshape(-1,3,HISTORY); h=self.temporal(temporal_in).reshape(len(hist),N_ROADS,-1)
        ids=torch.arange(N_ROADS,device=hist.device); h=h+self.road_proj(self.road_embedding(ids))[None]
        raw_global=torch.cat([hist.mean(1),hist.std(1,unbiased=False),(hist==0).to(hist.dtype).mean(1),hist[:,ANCHOR_IDX,-1]],1); g=self.global_encoder((raw_global-self.global_mean)/self.global_scale); phase_pred=torch.tanh(self.phase_head(g))
        film_input=torch.cat([g,phase_pred],1) if self.config.use_phase else g; gamma,beta=self.film(film_input).chunk(2,1); h=h*(1+.15*torch.tanh(gamma[:,None]))+.15*beta[:,None]
        if self.config.use_clause:
            c=self.clause_encoder(torch.cat([self.location_embedding(loc),self.type_embedding(typ)],-1)); score=torch.einsum('rd,bcd->brc',self.road_query.weight,self.clause_key(c))/4.; score=score.masked_fill((loc==0)[:,None,:],-1e4); attn=torch.softmax(score,-1); context=torch.einsum('brc,bch->brh',attn,self.clause_value(c)); h=h+torch.tanh(self.clause_scale)*context
        for block in self.graph_blocks:h=block(h,self.src,self.dst,self.geo_src,self.geo_dst)
        residual=torch.cat([head(h) for head in self.heads],-1)*self.road_std; return base+residual,phase_pred

CONFIGS=[ModelConfig('global_tcn_diffusion',False,False,False,0.),ModelConfig('phase_tcn_diffusion',True,False,False,1.0),ModelConfig('phase_clause_geo_diffusion',True,True,True,1.0)]
display(pd.DataFrame([asdict(x) for x in CONFIGS]))


## 7. Training utilities dan three-fold OOF ablation

Fold model memakai fixed epoch count. Tabel menyimpan overall, per-horizon, per-block, dan phase error. FAST_MODE hanya smoke test; scorenya tidak comparable.


In [ ]:
def make_loader(times,shuffle,seed):
    g=torch.Generator(); g.manual_seed(seed); return DataLoader(TrafficDataset(times),batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),generator=g)
def horizon_mse(y,p):return np.array([np.mean((y[...,i]-p[...,i])**2) for i in range(3)])
def circular_phase_error(true_phase,pred_phase):
    a=np.arctan2(true_phase[:,0],true_phase[:,1]); b=np.arctan2(pred_phase[:,0],pred_phase[:,1]); d=np.abs(a-b); d=np.minimum(d,2*np.pi-d); return d*STEPS_PER_DAY/(2*np.pi)*4

def evaluate_model(model,loader,desc):
    model.eval(); yy=[]; pp=[]; bb=[]; bases=[]; phases=[]; phase_preds=[]
    with torch.no_grad():
        for hist,loc,typ,target,phase,block in tqdm(loader,desc=desc,leave=False):
            pred,phase_pred=model(hist.to(DEVICE),loc.to(DEVICE),typ.to(DEVICE)); yy.append(target.numpy()); pp.append(pred.cpu().numpy()); bb.append(block.numpy()); bases.append(hist[:,:,-3:].mean(-1).numpy()); phases.append(phase.numpy()); phase_preds.append(phase_pred.cpu().numpy())
    return tuple(np.concatenate(x) for x in [yy,pp,bb,bases,phases,phase_preds])

def train_model(config,train_times,val_times,context,seed,epochs):
    set_seed(seed); model=PhaseClauseDiffusionNet(config,*context).to(DEVICE); train_loader=make_loader(train_times,True,seed); val_loader=make_loader(val_times,False,seed) if val_times is not None else None
    opt=torch.optim.AdamW(model.parameters(),lr=1.5e-3,weight_decay=1e-4); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,epochs)); logs=[]
    for epoch in trange(1,epochs+1,desc=f'{config.name} seed={seed}',leave=False):
        model.train(); sse=0.; n=0
        for hist,loc,typ,target,phase,_ in train_loader:
            hist,loc,typ,target,phase=hist.to(DEVICE),loc.to(DEVICE),typ.to(DEVICE),target.to(DEVICE),phase.to(DEVICE); opt.zero_grad(set_to_none=True); pred,phase_pred=model(hist,loc,typ)
            mse=((pred-target)**2).mean(); phase_loss=((phase_pred-phase)**2).mean(); loss=mse+config.phase_weight*phase_loss; loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.); opt.step(); sse+=mse.item()*target.numel(); n+=target.numel()
        sched.step()
        if val_loader is not None:
            yv,pv,_,_,tv,qv=evaluate_model(model,val_loader,'epoch validation'); val_mse=horizon_mse(yv,pv).mean(); phase_minutes=np.median(circular_phase_error(tv,qv))
        else: val_mse=np.nan; phase_minutes=np.nan
        logs.append({'epoch':epoch,'train_mse':sse/n,'val_mse':val_mse,'phase_median_min':phase_minutes})
    return model,pd.DataFrame(logs)

oof_predictions={}; cv_rows=[]; training_logs={}; shared_targets=[]; shared_blocks=[]; shared_bases=[]; shared_folds=[]
for ci,config in enumerate(tqdm(CONFIGS,desc='V4 architecture OOF')):
    pred_parts=[]
    for fold in range(N_FOLDS):
        context=fit_fold_context(fold_train_times[fold]); model,log=train_model(config,fold_train_times[fold],fold_val_times[fold],context,BASE_SEED+fold,CV_EPOCHS); loader=make_loader(fold_val_times[fold],False,BASE_SEED)
        y,p,b,base,tp,qp=evaluate_model(model,loader,f'{config.name} fold {fold} OOF'); pred_parts.append(p.astype(np.float32)); training_logs[(config.name,fold)]=log
        scores=horizon_mse(y,p); phase_err=np.median(circular_phase_error(tp,qp)); cv_rows.append({'model':config.name,'fold':fold,'h5':scores[0],'h10':scores[1],'h15':scores[2],'mean':scores.mean(),'m1':np.mean((y[b==0]-p[b==0])**2),'m2':np.mean((y[b==1]-p[b==1])**2),'phase_median_min':phase_err})
        if ci==0: shared_targets.append(y.astype(np.float32)); shared_blocks.append(b); shared_bases.append(base.astype(np.float32)); shared_folds.append(np.full(len(y),fold,np.int8))
        del model; gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    oof_predictions[config.name]=np.concatenate(pred_parts)

oof_y=np.concatenate(shared_targets); oof_block=np.concatenate(shared_blocks); oof_base=np.concatenate(shared_bases); oof_fold=np.concatenate(shared_folds)
cv_report=pd.DataFrame(cv_rows); summary=cv_report.groupby('model').agg(h5=('h5','mean'),h10=('h10','mean'),h15=('h15','mean'),mean=('mean','mean'),fold_std=('mean','std'),m1=('m1','mean'),m2=('m2','mean'),phase_median_min=('phase_median_min','median')).sort_values('mean'); display(summary.round(4)); display(cv_report.round(4))
print('OOF rows:',len(oof_y),'block proportions:',np.bincount(oof_block)/len(oof_block),'| FAST_MODE scores are not comparable.' if FAST_MODE else '')

fig,ax=plt.subplots(1,3,figsize=(18,4))
summary[['h5','h10','h15']].plot(kind='bar',ax=ax[0],title='Natural OOF MSE'); ax[0].tick_params(axis='x',rotation=25); ax[0].set_ylabel('MSE')
for (name,fold),log in training_logs.items(): ax[1].plot(log.epoch,log.val_mse,label=f'{name}/f{fold}',alpha=.75)
ax[1].set(title='Fold validation curves (fixed epoch)',xlabel='epoch',ylabel='MSE'); ax[1].legend(fontsize=6,ncol=2)
cv_report.boxplot(column='mean',by='model',ax=ax[2]); ax[2].set_title('Fold stability'); ax[2].tick_params(axis='x',rotation=25); plt.suptitle(''); plt.tight_layout(); plt.show()


## 8. Cross-fitted OOF ensemble

Candidates adalah mean-last-3 dan ketiga neural configurations. Untuk setiap held fold, simplex weights dipelajari dari fold lain. `crossfit_MSE` adalah estimasi ensemble yang fair; `final weights` dipelajari dari seluruh OOF hanya untuk refit/test.


In [ ]:
candidate_names=['mean3']+[c.name for c in CONFIGS]; candidate_oof=[np.repeat(oof_base[...,None],3,axis=-1)]+[oof_predictions[c.name] for c in CONFIGS]
def sufficient_stats(y,preds,sample_mask,h,batch=64):
    ids=np.flatnonzero(sample_mask); G=np.zeros((len(preds),len(preds)),np.float64); c=np.zeros(len(preds),np.float64); total=0
    for start in range(0,len(ids),batch):
        ix=ids[start:start+batch]; X=np.stack([p[ix,:,h].reshape(-1) for p in preds],1).astype(np.float64); yy=y[ix,:,h].reshape(-1).astype(np.float64); G+=X.T@X; c+=X.T@yy; total+=len(yy)
    return G/total,c/total
def solve_weights(y,preds,mask,h):
    G,c=sufficient_stats(y,preds,mask,h); fun=lambda w:float(w@G@w-2*c@w); jac=lambda w:2*G@w-2*c
    result=minimize(fun,np.full(len(preds),1/len(preds)),jac=jac,bounds=[(0,1)]*len(preds),constraints={'type':'eq','fun':lambda w:w.sum()-1},method='SLSQP',options={'ftol':1e-11,'maxiter':300}); assert result.success,result.message; return result.x

crossfit_pred=np.empty_like(oof_y); crossfit_rows=[]
for fold in range(N_FOLDS):
    train_mask=oof_fold!=fold; val_mask=oof_fold==fold
    for h in range(3):
        w=solve_weights(oof_y,candidate_oof,train_mask,h); crossfit_pred[val_mask,:,h]=sum(w[j]*candidate_oof[j][val_mask,:,h] for j in range(len(w))); crossfit_rows.append({'held_fold':fold,'horizon':f'h{HORIZONS[h]}',**{f'w_{n}':w[j] for j,n in enumerate(candidate_names)}})
crossfit_score=horizon_mse(oof_y,crossfit_pred)
final_weights=np.stack([solve_weights(oof_y,candidate_oof,np.ones(len(oof_y),bool),h) for h in range(3)])
weight_report=pd.DataFrame(final_weights,columns=candidate_names,index=[f'h{h}' for h in HORIZONS]); display(pd.DataFrame(crossfit_rows).round(4)); display(weight_report.round(5))
print('Cross-fitted ensemble MSE:',crossfit_score.round(5),'mean',crossfit_score.mean())

fig,ax=plt.subplots(1,2,figsize=(14,4)); pd.DataFrame(crossfit_rows).set_index(['held_fold','horizon'])[[f'w_{n}' for n in candidate_names]].plot(kind='bar',stacked=True,ax=ax[0],title='Cross-fit weights'); ax[0].legend(fontsize=6); weight_report.plot(kind='bar',stacked=True,ax=ax[1],title='Final all-OOF weights',ylim=(0,1)); ax[1].tick_params(axis='x',rotation=0); plt.tight_layout(); plt.show()

summary.reset_index().to_csv(OUTPUT_DIR/'v4_cv_results.csv',index=False); pd.DataFrame(crossfit_rows).to_csv(OUTPUT_DIR/'v4_crossfit_weights.csv',index=False); weight_report.to_csv(OUTPUT_DIR/'v4_final_weights.csv')


## 9. Full refit dan test inference

Hanya neural candidate dengan final OOF weight >2% yang direfit. Semua valid continuous origins dipakai; dua/lebih seed dirata-ratakan. Ini menjaga runtime proporsional terhadap evidence OOF.


In [ ]:
full_times=[np.arange(HISTORY-1,len(z)-max(HORIZONS),FULL_STRIDE,dtype=np.int32) for z in train_speeds]
if FAST_MODE: full_times=[x[-min(18,len(x)):] for x in full_times]
full_context=fit_fold_context(full_times)
active_names=[c.name for j,c in enumerate(CONFIGS,1) if final_weights[:,j].max()>.05]
if not active_names: active_names=[summary.index[0]]
print('Active neural candidates for refit:',active_names)

class TestDataset(Dataset):
    def __len__(self):return len(test_hist)
    def __getitem__(self,i):return torch.from_numpy(np.asarray(test_hist[i],np.float32).T.copy()),torch.from_numpy(test_loc[i]),torch.from_numpy(test_type[i])
test_loader=DataLoader(TestDataset(),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
test_predictions={'mean3':np.repeat(test_hist[:,-3:].mean(1)[...,None],3,axis=-1).astype(np.float32)}
for config in tqdm(CONFIGS,desc='Full model candidates'):
    if config.name not in active_names: continue
    seed_parts=[]
    for seed in tqdm(FINAL_SEEDS,desc=f'{config.name} seeds',leave=False):
        model,_=train_model(config,full_times,None,full_context,seed,FINAL_EPOCHS); model.eval(); parts=[]
        with torch.no_grad():
            for hist,loc,typ in tqdm(test_loader,desc=f'test {config.name}/{seed}',leave=False):
                pred,_=model(hist.to(DEVICE),loc.to(DEVICE),typ.to(DEVICE)); parts.append(pred.cpu().numpy())
        seed_parts.append(np.concatenate(parts).astype(np.float32)); torch.save({'config':asdict(config),'seed':seed,'epochs':FINAL_EPOCHS,'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()}},OUTPUT_DIR/f'v4_{config.name}_seed{seed}.pt')
        del model; gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    test_predictions[config.name]=np.mean(seed_parts,axis=0).astype(np.float32)

# Renormalize weights if a negligible candidate was intentionally skipped.
available=np.array([n=='mean3' or n in test_predictions for n in candidate_names])
W=final_weights.copy(); W[:,~available]=0; W=W/W.sum(1,keepdims=True)
test_pred=np.empty_like(test_predictions['mean3'])
for h in range(3): test_pred[...,h]=sum(W[h,j]*test_predictions[n][...,h] for j,n in enumerate(candidate_names) if n in test_predictions)
test_pred=np.clip(test_pred,0,160).astype(np.float32)
print('Test prediction:',test_pred.shape,'range',float(test_pred.min()),float(test_pred.max()),'mean',float(test_pred.mean()))


## 10. Submission, diagnostics, dan interpretation

Audit boundary ID memastikan flatten order sample → horizon → road. Download submission, CV tables, final weights, diagnostics, checkpoint, dan executed notebook.


In [ ]:
submission=sample_submission.copy(); flat=test_pred.transpose(0,2,1).reshape(-1)
assert len(flat)==len(submission); assert submission.id.iloc[0]=='test_00000_h5_r0'; assert submission.id.iloc[N_ROADS]=='test_00000_h10_r0'; assert submission.id.iloc[3*N_ROADS]=='test_00001_h5_r0'; assert submission.id.iloc[-1]=='test_00539_h15_r1259'
submission['speed']=flat; assert submission.id.nunique()==len(submission) and submission.speed.notna().all() and submission.speed.between(0,160).all()
submission_path=OUTPUT_DIR/'submission_v4.csv'; submission.to_csv(submission_path,index=False)

audit=[]; mean3=test_hist[:,-3:].mean(1)
for i,h in enumerate(HORIZONS):
    x=test_pred[...,i]; audit.append({'horizon':f'h{h}','mean':x.mean(),'std':x.std(),'min':x.min(),'max':x.max(),'zeros':int(np.sum(x==0)),'mean_abs_vs_mean3':np.abs(x-mean3).mean()})
audit_df=pd.DataFrame(audit); display(audit_df.round(5))
fig,ax=plt.subplots(1,3,figsize=(17,4))
for i,h in enumerate(HORIZONS):ax[0].hist(test_pred[...,i].ravel(),bins=70,alpha=.4,label=f'h{h}')
ax[0].set(title='V4 submission distributions',xlabel='km/h'); ax[0].legend()
ax[1].hist((test_pred[...,0]-test_pred[...,1]).ravel(),bins=70,alpha=.5,label='h5-h10'); ax[1].hist((test_pred[...,1]-test_pred[...,2]).ravel(),bins=70,alpha=.5,label='h10-h15'); ax[1].set(title='Horizon deltas'); ax[1].legend()
ax[2].scatter(mean3.ravel(),test_pred[...,2].ravel(),s=1,alpha=.08); ax[2].plot([0,120],[0,120],'r--'); ax[2].set(title='h15 vs mean-last-3',xlabel='mean3',ylabel='V4 h15'); plt.tight_layout(); plt.show()

diagnostics={'public_v3':PUBLIC_V3,'cv_folds':N_FOLDS,'cv_stride':CV_STRIDE,'cv_epochs':CV_EPOCHS,'crossfit_mse_by_horizon':crossfit_score.tolist(),'crossfit_mean':float(crossfit_score.mean()),'candidate_names':candidate_names,'final_weights':W.tolist(),'active_models':active_names,'final_seeds':FINAL_SEEDS,'submission_rows':len(submission)}
(OUTPUT_DIR/'v4_diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
print('Saved:',submission_path); print('Also saved: v4_cv_results.csv, v4_crossfit_weights.csv, v4_final_weights.csv, v4_diagnostics.json, checkpoints')
display(pd.concat([submission.head(3),submission.tail(3)]))


## 11. Decision rules

- Jangan bandingkan FAST_MODE score atau old-tail 26.22 dengan public leaderboard.
- Kandidat dianggap membantu hanya jika natural OOF mean dan fold stability membaik.
- `crossfit_mean` adalah score lokal utama V4; all-OOF final weight hanya digunakan untuk test inference.
- Jika corrected OOF masih sekitar high-30, public area 20 belum terbukti. Submit V4 sebagai entry terpisah, lalu bandingkan terhadap V3 38.639.
- Jika phase membantu tetapi clause/geometry tidak, notebook otomatis memberi bobot lebih besar ke model phase yang lebih sederhana.

Expected Kaggle outputs:

```text
/kaggle/working/submission_v4.csv
/kaggle/working/v4_cv_results.csv
/kaggle/working/v4_crossfit_weights.csv
/kaggle/working/v4_final_weights.csv
/kaggle/working/v4_diagnostics.json
/kaggle/working/v4_<active_model>_seed*.pt
```
